In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.read_csv("used_cars_cleaned_08.csv")

# Explicitly select information available before estimating the price.
feature_cols = [
    "make", "model", "version", "mileage_miles", "reg_year",
    "fuel", "transmission", "power_hp", "body_type", "body_color",
    "doors", "seats", "previous_owners", "drivetrain", "emission_class",
    "annual_tax_gbp", "seller_type", "seller_city", "seller_county",
    "co2_gmile",
]

X = df.loc[:, feature_cols].copy()
y = df["price_gbp"].copy()

# Keep identical predictor rows together, even if their prices differ.
groups = pd.util.hash_pandas_object(X, index=False)

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(splitter.split(X, y, groups=groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
train_groups = groups.iloc[train_idx]

assert not groups.iloc[val_idx].isin(train_groups).any()

# Precompute folds for the later searches. These positions refer to X_train.
# X_train_text also works with these folds if it preserves the same row order.
grouped_cv = list(
    GroupKFold(n_splits=3).split(X_train, y_train, groups=train_groups)
)
cv = grouped_cv

numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols),
])

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression()),
])

linear_model.fit(X_train, y_train)

train_predictions = linear_model.predict(X_train)
val_predictions = linear_model.predict(X_val)

results = []
for split, actual, predicted in [
    ("Training", y_train, train_predictions),
    ("Validation", y_val, val_predictions),
]:
    results.append({
        "Dataset": split,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "R²": r2_score(actual, predicted),
    })

print(f"Training listings: {len(X_train):,}")
print(f"Validation listings: {len(X_val):,}")
display(pd.DataFrame(results).round(3))

comparison = pd.DataFrame({
    "Actual Price": y_val.to_numpy(),
    "Predicted Price": val_predictions,
}, index=y_val.index)
comparison["Absolute Error"] = (
    comparison["Actual Price"] - comparison["Predicted Price"]
).abs()

display(comparison.head(10).round(2))
print("Negative validation predictions:", (val_predictions < 0).sum())

Training listings: 289,266
Validation listings: 72,324


,Dataset,MAE (£),RMSE (£),R²
0,Training,1605.726,7824.662,0.945
1,Validation,2180.196,13021.904,0.791


,Actual Price,Predicted Price,Absolute Error
6,26499,27689.67,1190.67
17,1499,-3319.76,4818.76
21,2795,4724.49,1929.49
27,3795,3627.68,167.32
28,7800,9112.03,1312.03
29,16895,17334.75,439.75
32,5490,7418.85,1928.85
33,41388,41070.95,317.05
38,3995,5266.42,1271.42
41,13168,12908.88,259.12


Negative validation predictions: 1728


In [2]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge(solver="lsqr"))
])

param_grid = {
    "model__alpha": [0.1, 1, 10, 100, 1000]
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)

search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    verbose=1,
    refit=True
)

# Tune using training data only
search.fit(X_train, y_train)

best_ridge = search.best_estimator_

print("Best parameters:", search.best_params_)
print("Best cross-validation MAE (£):", round(-search.best_score_, 2))

# Evaluate the selected model on validation data
predictions = best_ridge.predict(X_val)

print("\nValidation results")
print("MAE (£):", round(mean_absolute_error(y_val, predictions), 2))
print("RMSE (£):", round(np.sqrt(
    mean_squared_error(y_val, predictions)
), 2))
print("R²:", round(r2_score(y_val, predictions), 4))

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best parameters: {'model__alpha': 0.1}
Best cross-validation MAE (£): 2312.37

Validation results
MAE (£): 2235.42
RMSE (£): 13071.57
R²: 0.7889


In [3]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Exclude missing-value indicators from polynomial expansion
poly_cols = [
    col for col in numeric_cols
    if not col.endswith("_missing")
]

indicator_cols = [
    col for col in numeric_cols
    if col.endswith("_missing")
]

polynomial_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("polynomial", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler())
])

poly_preprocessor = ColumnTransformer([
    ("numeric_poly", polynomial_pipeline, poly_cols),
    ("indicators", "passthrough", indicator_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

poly_model = Pipeline([
    ("preprocessor", poly_preprocessor),
    ("model", Ridge(alpha=100, solver="lsqr"))
])

# Use the same split as previous models for a fair comparison
poly_model.fit(X_train, y_train)

results = []

for name, X_part, y_part in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val)
]:
    predictions = poly_model.predict(X_part)

    results.append({
        "Dataset": name,
        "MAE (£)": mean_absolute_error(y_part, predictions),
        "RMSE (£)": np.sqrt(mean_squared_error(y_part, predictions)),
        "R²": r2_score(y_part, predictions)
    })

display(pd.DataFrame(results).round(3))

,Dataset,MAE (£),RMSE (£),R²
0,Training,3455.490,22311.723,0.554
1,Validation,3448.116,15361.762,0.708


In [4]:
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import numpy as np

param_grid = {
    "preprocessor__numeric_poly__polynomial__degree": [1, 2],
    "model__alpha": [0.1, 1, 10, 100, 1000]
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)

poly_search = GridSearchCV(
    estimator=poly_model,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1,
    refit=True,
    error_score="raise"
)

# Tune on training data only
poly_search.fit(X_train, y_train)

best_poly_model = poly_search.best_estimator_

print("Best parameters:", poly_search.best_params_)
print("Best CV MAE (£):", round(-poly_search.best_score_, 2))

# Compare training and validation performance
results = []

for name, X_part, y_part in [
    ("Training", X_train, y_train),
    ("Validation", X_val, y_val)
]:
    predictions = best_poly_model.predict(X_part)

    results.append({
        "Dataset": name,
        "MAE (£)": mean_absolute_error(y_part, predictions),
        "RMSE (£)": np.sqrt(mean_squared_error(y_part, predictions)),
        "R²": r2_score(y_part, predictions)
    })

display(pd.DataFrame(results).round(3))

# Show all parameter combinations, best first
tuning_results = pd.DataFrame(poly_search.cv_results_)
tuning_results["CV MAE (£)"] = -tuning_results["mean_test_score"]

display(
    tuning_results[[
        "param_preprocessor__numeric_poly__polynomial__degree",
        "param_model__alpha",
        "CV MAE (£)",
        "rank_test_score"
    ]]
    .sort_values("rank_test_score")
    .reset_index(drop=True)
)

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best parameters: {'model__alpha': 1, 'preprocessor__numeric_poly__polynomial__degree': 2}
Best CV MAE (£): 2253.29


,Dataset,MAE (£),RMSE (£),R²
0,Training,2131.245,9827.773,0.914
1,Validation,2375.482,12461.338,0.808


,param_preprocessor__numeric_poly__polynomial__degree,param_model__alpha,CV MAE (£),rank_test_score
0,2,1.0,2253.292730,1
1,1,0.1,2310.697930,2
2,1,1.0,2462.604668,3
3,2,10.0,2857.167125,4
4,2,0.1,3077.160683,5
5,1,10.0,3288.404575,6
6,2,100.0,3687.594636,7
7,2,1000.0,4489.838420,8
8,1,100.0,4609.627768,9
9,1,1000.0,5984.967891,10


In [5]:
from sklearn.linear_model import SGDRegressor
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Use the polynomial degree selected by your previous search
sgd_model = Pipeline([
    ("preprocessor", best_poly_model.named_steps["preprocessor"]),
    ("model", SGDRegressor(
        loss="squared_error",
        penalty="l2",
        max_iter=3000,
        tol=1e-4,
        average=True,
        random_state=42
    ))
])

param_grid = {
    "model__alpha": [0.0001, 0.001, 0.01],
    "model__learning_rate": ["adaptive"],
    "model__eta0": [0.0001, 0.001]
}

sgd_search = GridSearchCV(
    sgd_model,
    param_grid,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1
)

sgd_search.fit(X_train, y_train)

predictions = sgd_search.predict(X_val)

print("Best parameters:", sgd_search.best_params_)
print("Validation MAE (£):",
      round(mean_absolute_error(y_val, predictions), 2))
print("Validation RMSE (£):",
      round(np.sqrt(mean_squared_error(y_val, predictions)), 2))
print("Validation R²:", round(r2_score(y_val, predictions), 4))

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best parameters: {'model__alpha': 0.0001, 'model__eta0': 0.001, 'model__learning_rate': 'adaptive'}
Validation MAE (£): 3114.28
Validation RMSE (£): 14794.66
Validation R²: 0.7296


In [6]:
from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

log_ridge = TransformedTargetRegressor(
    regressor=Pipeline([
        ("preprocessor", clone(preprocessor)),
        ("model", Ridge(solver="lsqr"))
    ]),
    func=np.log1p,
    inverse_func=np.expm1
)

log_search = GridSearchCV(
    log_ridge,
    param_grid={
        "regressor__model__alpha": [0.01, 0.1, 1, 10, 100]
    },
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1
)

log_search.fit(X_train, y_train)

# Predictions are already in pounds
predictions = log_search.predict(X_val)

print("Best parameters:", log_search.best_params_)
print("Validation MAE (£):",
      round(mean_absolute_error(y_val, predictions), 2))
print("Validation RMSE (£):",
      round(np.sqrt(mean_squared_error(y_val, predictions)), 2))
print("Validation R²:", round(r2_score(y_val, predictions), 4))

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best parameters: {'regressor__model__alpha': 0.1}
Validation MAE (£): 1610.46
Validation RMSE (£): 12257.21
Validation R²: 0.8144


In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Work on copies of the existing split
X_train_text = X_train.copy()
X_val_text = X_val.copy()

for frame in [X_train_text, X_val_text]:
    frame["version"] = frame["version"].fillna("").astype(str)

numeric_cols = X_train_text.select_dtypes(
    include="number"
).columns.tolist()

# Exclude version: it will be handled by CountVectorizer
other_categorical_cols = [
    col for col in X_train_text.select_dtypes(
        include=["object", "string", "category"]
    ).columns
    if col != "version"
]

text_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), numeric_cols),

    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), other_categorical_cols),

    # A string selector passes a 1D text column to CountVectorizer
    ("version_text", CountVectorizer(
        lowercase=True,
        strip_accents="unicode",
        token_pattern=r"(?u)\b\w[\w.-]*\b",
        min_df=3,
        max_features=15000
    ), "version")
])

text_model = Pipeline([
    ("preprocessor", text_preprocessor),
    ("model", Ridge(solver="lsqr"))
])

text_search = GridSearchCV(
    text_model,
    param_grid={
        "preprocessor__version_text__ngram_range": [(1, 1), (1, 2)],
        "model__alpha": [0.01, 0.1, 1, 10]
    },
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=2,
    pre_dispatch=2,
    verbose=1
)

text_search.fit(X_train_text, y_train)

predictions = text_search.predict(X_val_text)

print("Best parameters:", text_search.best_params_)
print("Validation MAE (£):",
      round(mean_absolute_error(y_val, predictions), 2))
print("Validation RMSE (£):",
      round(np.sqrt(mean_squared_error(y_val, predictions)), 2))
print("Validation R²:", round(r2_score(y_val, predictions), 4))

Fitting 3 folds for each of 8 candidates, totalling 24 fits
Best parameters: {'model__alpha': 1, 'preprocessor__version_text__ngram_range': (1, 2)}
Validation MAE (£): 2491.25
Validation RMSE (£): 13184.33
Validation R²: 0.7853


In [8]:
err = (y_val - best_ridge.predict(X_val)).abs().sort_values(ascending=False)
X_val.loc[err.index[:20]].assign(actual=y_val, err=err)

,make,model,version,mileage_miles,reg_year,fuel,transmission,power_hp,body_type,body_color,...,previous_owners,drivetrain,emission_class,annual_tax_gbp,seller_type,seller_city,seller_county,co2_gmile,actual,err
270550,Aston Martin,Valour,5.2 V12 BiTurbo Euro 6 (s/s) 2dr,1100,2024,Petrol,Manual,715,Coupe,Unknown,...,1,Rear Wheel Drive,Euro 6,200.0,Trade,Pangbourne,West Berkshire,547,1785000,1.614212e+06
20590,Porsche,911,993 GT2,21727,1995,Petrol,Manual,444,Coupe,Silver,...,3,Rear Wheel Drive,Euro 1,200.0,Trade,Westminster,Middlesex,497,1600000,1.493052e+06
311536,Aston Martin,Valour,5.2 V12 BiTurbo Euro 6 (s/s) 2dr,250,2025,Petrol,Manual,715,Coupe,Black,...,1,Rear Wheel Drive,Euro 6,200.0,Trade,Preston,Lancashire,547,1499995,1.338091e+06
113656,Rimac,Nevera,120kWh GT Auto 4WD 2dr,735,2023,Electric,Automatic,1888,Coupe,Green,...,1,Four Wheel Drive,Euro 6,200.0,Trade,London,London,0,1500000,1.313897e+06
36377,Aston Martin,Valhalla,Unknown,193,2026,Petrol Plug-in Hybrid,Automatic,265,Unknown,Yellow,...,1,Unknown,Euro 6,200.0,Trade,Newtownabbey,County Antrim,63,1099995,8.887975e+05
280161,Rolls-Royce,Phantom,6.7 V12 Drophead Coupe Auto Euro 6 2dr,479,2017,Petrol,Automatic,460,Convertible,Red,...,2,Rear Wheel Drive,Euro 6,200.0,Trade,London,Islington,558,849000,5.514217e+05
208083,Ferrari,599,6.0 V12 GTB Fiorano Coupe 2dr Petrol F1 Euro 4...,22700,2009,Petrol,Automatic,612,Coupe,Red,...,2,Rear Wheel Drive,Euro 4,790.0,Private,Finsbury Park,Haringey,789,159000,3.588409e+05
166650,Ferrari,488 Pista,3.9T V8 Spider F1 DCT Euro 6 (s/s) 2dr,1000,2019,Petrol,Automatic,710,Convertible,Red,...,2,Rear Wheel Drive,Euro 6,200.0,Trade,Huddersfield,West Yorkshire,465,999995,3.378904e+05
343864,Rolls-Royce,Ghost,6.6 V12 Auto Euro 6 4dr,22000,2015,Petrol,Automatic,563,Saloon,Blue,...,4,Rear Wheel Drive,Euro 6,790.0,Private,Barnsley,South Yorkshire,526,420000,3.187146e+05
74588,Rolls-Royce,Wraith,6.6 V12 Black Badge Auto Euro 6 2dr,1200,2023,Petrol,Automatic,591,Coupe,Black,...,1,Rear Wheel Drive,Euro 6,200.0,Trade,Wilmslow,Cheshire,578,495000,2.984453e+05


In [9]:
from sklearn.base import clone
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

svd_linear = Pipeline([
    ("preprocessor", clone(text_preprocessor)),
    ("reduction", TruncatedSVD(random_state=42)),
    ("model", LinearRegression())
])

# Dimensionality reduction can be expensive: start small
svd_search = GridSearchCV(
    svd_linear,
    param_grid={
        "reduction__n_components": [50, 100, 200]
    },
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=1,
    verbose=1,
    error_score="raise"
)

svd_search.fit(X_train_text, y_train)

best_svd_model = svd_search.best_estimator_
predictions = best_svd_model.predict(X_val_text)

print("Best parameters:", svd_search.best_params_)
print("Validation MAE (£):",
      round(mean_absolute_error(y_val, predictions), 2))
print("Validation RMSE (£):",
      round(np.sqrt(mean_squared_error(y_val, predictions)), 2))
print("Validation R²:", round(r2_score(y_val, predictions), 4))

variance = (
    best_svd_model.named_steps["reduction"]
    .explained_variance_ratio_.sum()
)
print("Feature variance retained:", f"{variance:.1%}")

Fitting 3 folds for each of 3 candidates, totalling 9 fits
Best parameters: {'reduction__n_components': 100}
Validation MAE (£): 6707.07
Validation RMSE (£): 20602.26
Validation R²: 0.4757
Feature variance retained: 80.5%


In [10]:
from pathlib import Path

import joblib
from sklearn.utils.validation import check_is_fitted

check_is_fitted(linear_model)

model_path = Path("linear_reg.joblib")
joblib.dump(linear_model, model_path, compress=3)

print(f"Saved: {model_path.resolve()}")

Saved: D:\Msc DS\Machine Learning\Project\Sample Project\linear_reg.joblib


In [2]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline


CSV_PATH = Path(
    "uk_used_cars_autotrader.csv"
)
MODEL_PATH = Path(
    "used_car_log_price_ridge.joblib"
)
SEED = 42

for path in (CSV_PATH, MODEL_PATH):
    if not path.is_file():
        raise FileNotFoundError(f"File not found: {path}")

# Preserve the column order used to construct the original grouping hashes.
numeric_columns = [
    "mileage_miles", "reg_year", "power_hp", "doors", "seats",
    "previous_owners", "annual_tax_gbp", "co2_gmile",
]
categorical_columns = [
    "make", "model", "version", "fuel", "transmission", "body_type",
    "body_color", "drivetrain", "emission_class", "seller_type",
    "seller_city", "seller_county",
]


def prepare_listings(data):
    eligible = (
        data["condition"].eq("Used")
        & data["reg_year"].notna()
        & (data["previous_owners"].isna() | data["previous_owners"].lt(8))
    )
    data = data.loc[eligible].copy()

    co2 = (data["co2_gkm"] * 1.609344).round()
    data["co2_gmile"] = co2.mask(co2.isin([1608, 2092]))
    data["power_hp"] = data["power_hp"].mask(data["power_hp"].le(0))

    for column in ["doors", "seats"]:
        data[column] = data[column].mask(data[column].lt(2))

    for column in categorical_columns:
        data[column] = (
            data[column].str.strip()
            .replace({"": "Unknown", "Unlisted": "Unknown"})
            .fillna("Unknown")
        )

    return data[numeric_columns + categorical_columns], data["price_gbp"]


saved_ridge = joblib.load(MODEL_PATH)["model"]
raw_data = pd.read_csv(CSV_PATH)

# Recreate the original split while reserving the original test rows.
development, _ = train_test_split(raw_data, test_size=0.20, random_state=SEED)
X_dev, y_dev = prepare_listings(development)

groups = pd.util.hash_pandas_object(X_dev, index=False)
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_idx, val_idx = next(splitter.split(X_dev, y_dev, groups=groups))

X_dev = X_dev.assign(
    log_mileage=np.log1p(X_dev["mileage_miles"]),
    make_model=X_dev["make"] + " | " + X_dev["model"],
)
X_dev = X_dev.loc[:, saved_ridge.feature_names_in_]

X_train, X_val = X_dev.iloc[train_idx], X_dev.iloc[val_idx]
y_train, y_val = y_dev.iloc[train_idx], y_dev.iloc[val_idx]
train_groups = groups.iloc[train_idx]

base_prediction = saved_ridge.predict(X_val)
base_mae = mean_absolute_error(y_val, base_prediction)
print(f"Saved Ridge MAE on {len(X_val):,} validation listings: £{base_mae:,.2f}")

# Check the baseline before starting the five Ridge fits.
if len(X_val) != 73_695 or not np.isclose(
    base_mae, 1764.78, atol=0.05, rtol=0
):
    raise ValueError(
        "Expected 73,695 validation listings and approximately £1,764.78 MAE. "
        "Check that the raw CSV and model match the original Ridge run."
    )

# Each training row gets a prediction from a Ridge fit that held out its group.
oof_predictions = np.full(len(X_train), np.nan)
cv = GroupKFold(n_splits=5)

for fold, (fit_idx, score_idx) in enumerate(
    cv.split(X_train, y_train, groups=train_groups), start=1
):
    print(f"Fitting Ridge fold {fold}/5...", flush=True)
    fold_ridge = clone(saved_ridge)
    fold_ridge.fit(X_train.iloc[fit_idx], y_train.iloc[fit_idx])
    oof_predictions[score_idx] = fold_ridge.predict(X_train.iloc[score_idx])

if not np.isfinite(oof_predictions).all():
    raise ValueError("Some out-of-fold predictions are missing or invalid.")

training_residuals = y_train.to_numpy() - oof_predictions

# Fit a separate model to predict the signed error directly in pounds.
residual_model = Pipeline([
    (
        "preprocessor",
        clone(saved_ridge.regressor_.named_steps["preprocessor"]),
    ),
    (
        "model",
        GradientBoostingRegressor(
            loss="huber",
            n_estimators=40,
            learning_rate=0.05,
            max_depth=3,
            min_samples_leaf=30,
            random_state=SEED,
            verbose=1,
        ),
    ),
])

print("Fitting Gradient Boosting on Ridge residuals...", flush=True)
residual_model.fit(X_train, training_residuals)

predicted_correction = residual_model.predict(X_val)
corrected_prediction = base_prediction + predicted_correction

actual = y_val.to_numpy()
positive_price = actual > 0
results = []

for name, prediction in [
    ("Saved log-price Ridge", base_prediction),
    ("Ridge + Gradient Boosting correction", corrected_prediction),
]:
    residual = actual - prediction

    results.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, prediction),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, prediction)),
        "MAPE (%)": 100 * mean_absolute_percentage_error(
            actual[positive_price], prediction[positive_price]
        ),
        "R²": r2_score(actual, prediction),
        "Within 10% (%)": 100 * np.mean(
            np.abs(residual[positive_price]) <= 0.10 * actual[positive_price]
        ),
        "Mean residual (£)": residual.mean(),
    })

comparison = pd.DataFrame(results).set_index("Model")
display(comparison.round(3))

corrected_mae = mean_absolute_error(actual, corrected_prediction)
print(f"Validation MAE improvement: £{base_mae - corrected_mae:+,.2f}")

prediction_details = pd.DataFrame({
    "Actual (£)": actual,
    "Ridge prediction (£)": base_prediction,
    "Predicted correction (£)": predicted_correction,
    "Corrected prediction (£)": corrected_prediction,
    "Ridge residual (£)": actual - base_prediction,
    "Corrected residual (£)": actual - corrected_prediction,
}, index=y_val.index)

display(prediction_details.head(15).round(2))

Saved Ridge MAE on 73,695 validation listings: £1,764.78
Fitting Ridge fold 1/5...
Fitting Ridge fold 2/5...
Fitting Ridge fold 3/5...
Fitting Ridge fold 4/5...
Fitting Ridge fold 5/5...
Fitting Gradient Boosting on Ridge residuals...
      Iter       Train Loss   Remaining Time 
         1     3774934.9085            1.06m
         2     3766014.2030           59.16s
         3     3758513.0747           56.58s
         4     3751168.1158           54.96s
         5     3745331.8167           52.83s
         6     3740835.3784           50.93s
         7     3736643.6311           49.28s
         8     3731512.8700           47.64s
         9     3727545.7463           46.05s
        10     3724673.9004           44.67s
        20     3696016.0110           31.68s
        30     3673642.2342           16.41s
        40     3654385.3664            0.00s


,MAE (£),RMSE (£),MAPE (%),R²,Within 10% (%),Mean residual (£)
Model,,,,,,
Saved log-price Ridge,1764.782,17201.323,11.056,0.757,71.864,177.718
Ridge + Gradient Boosting correction,1735.310,17187.347,10.786,0.757,72.758,181.649


Validation MAE improvement: £+29.47


,Actual (£),Ridge prediction (£),Predicted correction (£),Corrected prediction (£),Ridge residual (£),Corrected residual (£)
203137,1500,3215.62,-166.96,3048.66,-1715.62,-1548.66
479580,17870,16362.40,51.15,16413.55,1507.60,1456.45
36807,43950,39621.96,-284.78,39337.18,4328.04,4612.82
248239,34000,38807.50,471.26,39278.75,-4807.50,-5278.75
189028,27995,27843.56,-172.87,27670.69,151.44,324.31
16580,5490,5813.67,61.26,5874.93,-323.67,-384.93
294858,16910,19766.08,-302.32,19463.75,-2856.08,-2553.75
101479,13168,13121.29,-13.67,13107.63,46.71,60.37
162294,4950,6176.02,-183.74,5992.28,-1226.02,-1042.28
20322,17180,16995.83,95.04,17090.88,184.17,89.12


In [3]:
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
)

mae_residual_model = clone(residual_model).set_params(
    model__loss="absolute_error",
    model__verbose=1,
)

print("Fitting Gradient Boosting with absolute-error loss...", flush=True)
mae_residual_model.fit(X_train, training_residuals)

actual = y_val.to_numpy()
positive = actual > 0

predictions = {
    "Saved Ridge": base_prediction,
    "Ridge + Huber correction": corrected_prediction,
    "Ridge + absolute-error correction": (
        base_prediction + mae_residual_model.predict(X_val)
    ),
}

rows = []
for name, predicted in predictions.items():
    error = np.abs(actual - predicted)
    rows.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "MAPE (%)": 100 * mean_absolute_percentage_error(
            actual[positive], predicted[positive]
        ),
        "Within 10% (%)": 100 * np.mean(
            error[positive] <= 0.10 * actual[positive]
        ),
    })

comparison = pd.DataFrame(rows).set_index("Model").sort_values("MAE (£)")
display(comparison.round(3))

Fitting Gradient Boosting with absolute-error loss...
      Iter       Train Loss   Remaining Time 
         1        1987.0737            1.25m
         2        1985.7379            1.17m
         3        1984.5241            1.13m
         4        1983.4158            1.12m
         5        1982.2971            1.12m
         6        1981.3467            1.07m
         7        1980.3307            1.05m
         8        1979.4929            1.06m
         9        1978.5359            1.02m
        10        1977.6497           59.02s
        20        1971.1992           35.24s
        30        1967.2971           16.92s
        40        1963.3431            0.00s


,MAE (£),RMSE (£),MAPE (%),Within 10% (%)
Model,,,,
Ridge + Huber correction,1735.310,17187.347,10.786,72.758
Ridge + absolute-error correction,1740.237,17196.785,10.779,72.895
Saved Ridge,1764.782,17201.323,11.056,71.864


In [4]:
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
)
from sklearn.tree import DecisionTreeRegressor

ada_residual_model = clone(residual_model).set_params(
    model=AdaBoostRegressor(
        estimator=DecisionTreeRegressor(
            max_depth=3,
            min_samples_leaf=30,
            random_state=42,
        ),
        n_estimators=40,
        learning_rate=0.05,
        loss="linear",
        random_state=42,
    )
)

print("Fitting AdaBoost on five-fold Ridge residuals...", flush=True)
ada_residual_model.fit(X_train, training_residuals)

ada_correction = ada_residual_model.predict(X_val)
ada_corrected_prediction = base_prediction + ada_correction

actual = y_val.to_numpy()
positive = actual > 0

predictions = {
    "Saved log-price Ridge": base_prediction,
    "Ridge + Gradient Boosting (Huber)": corrected_prediction,
    "Ridge + AdaBoost": ada_corrected_prediction,
}

rows = []
for name, predicted in predictions.items():
    absolute_error = np.abs(actual - predicted)

    rows.append({
        "Model": name,
        "MAE (£)": mean_absolute_error(actual, predicted),
        "RMSE (£)": np.sqrt(mean_squared_error(actual, predicted)),
        "MAPE (%)": 100 * mean_absolute_percentage_error(
            actual[positive], predicted[positive]
        ),
        "Within 10% (%)": 100 * np.mean(
            absolute_error[positive] <= 0.10 * actual[positive]
        ),
    })

comparison = pd.DataFrame(rows).set_index("Model").sort_values("MAE (£)")
display(comparison.round(3))

mae_gain = (
    comparison.loc["Ridge + Gradient Boosting (Huber)", "MAE (£)"]
    - comparison.loc["Ridge + AdaBoost", "MAE (£)"]
)
print(f"AdaBoost MAE improvement over Huber: £{mae_gain:+,.2f}")

Fitting AdaBoost on five-fold Ridge residuals...


,MAE (£),RMSE (£),MAPE (%),Within 10% (%)
Model,,,,
Ridge + Gradient Boosting (Huber),1735.310,17187.347,10.786,72.758
Saved log-price Ridge,1764.782,17201.323,11.056,71.864
Ridge + AdaBoost,2009.676,17859.604,12.193,70.659


AdaBoost MAE improvement over Huber: £-274.37


In [5]:
from pathlib import Path

import joblib
import numpy as np
import sklearn

from sklearn.metrics import mean_absolute_error
from sklearn.utils.validation import check_is_fitted

check_is_fitted(saved_ridge)
check_is_fitted(residual_model)

model_path = Path("used_car_ridge_gradient_boosting.joblib")

bundle = {
    "base_model": saved_ridge,
    "correction_model": residual_model,  # Fitted Gradient Boosting with Huber loss
    "correction_weight": 1.0,
    "feature_columns": list(saved_ridge.feature_names_in_),
    "sklearn_version": sklearn.__version__,
    "validation_mae_gbp": float(mean_absolute_error(y_val, corrected_prediction)),
}

joblib.dump(bundle, model_path, compress=3)

# Verify that reloading reproduces the fitted ensemble's predictions.
reloaded = joblib.load(model_path)
sample = X_val.loc[:, reloaded["feature_columns"]].iloc[:100]

reloaded_predictions = (
    reloaded["base_model"].predict(sample)
    + reloaded["correction_weight"] * reloaded["correction_model"].predict(sample)
)

np.testing.assert_allclose(reloaded_predictions, corrected_prediction[:len(sample)])

print(f"Saved and verified: {model_path.resolve()}")
print(f"Validation MAE: £{reloaded['validation_mae_gbp']:,.2f}")

Saved and verified: D:\Msc DS\Machine Learning\Project\Sample Project\used_car_ridge_gradient_boosting.joblib
Validation MAE: £1,735.31
